# 📹 Step 1: Extract Audio from Videos

This notebook extracts high-quality WAV audio files from video recordings stored in Google Drive.

**What it does:**
- Scans a folder of video files (`.mp4`, `.mkv`, `.mov`, `.avi`, `.wmv`, `.flv`)
- Extracts mono 16kHz PCM WAV audio using FFmpeg — optimized for ASR models
- Skips files that have already been processed (safe to re-run)

**Input:** A Google Drive folder containing video files  
**Output:** A Google Drive folder containing `.wav` files (one per video)

---
## Prerequisites
- Google Colab environment (FFmpeg is pre-installed)
- Video files uploaded to Google Drive


## ⚙️ Configuration
Edit the paths below to match your Google Drive folder structure.

In [ ]:
import os
import subprocess
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# ═══════════════════════════════════════════
# CONFIGURATION — Edit these paths
# ═══════════════════════════════════════════
INPUT_FOLDER = "/content/drive/MyDrive/your_videos/Day_1"      # Folder with your video files
OUTPUT_FOLDER = "/content/drive/MyDrive/your_audio_wavs/Day_1"  # Where WAV files will be saved
# ═══════════════════════════════════════════

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"Input:  {INPUT_FOLDER}")
print(f"Output: {OUTPUT_FOLDER}")


## 🎵 Extract Audio

Runs FFmpeg on each video to produce a 16kHz mono WAV file.  
These settings are optimized for speech recognition models like Qwen3-ASR:
- **16kHz sample rate** — native for most ASR models
- **Mono channel** — speech doesn't need stereo
- **PCM 16-bit** — lossless, maximum compatibility


In [ ]:
SUPPORTED_FORMATS = ('.mp4', '.mkv', '.mov', '.avi', '.wmv', '.flv')

files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(SUPPORTED_FORMATS)]
print(f"Found {len(files)} video(s) in input folder.\n")

for filename in files:
    input_path = os.path.join(INPUT_FOLDER, filename)
    output_filename = os.path.splitext(filename)[0] + ".wav"
    output_path = os.path.join(OUTPUT_FOLDER, output_filename)

    # Skip already-processed files
    if os.path.exists(output_path):
        print(f"[skip] {output_filename} (already exists)")
        continue

    print(f"[extracting] {filename}...")

    try:
        subprocess.run([
            'ffmpeg', '-i', input_path,
            '-vn',                  # No video
            '-acodec', 'pcm_s16le', # PCM 16-bit Little Endian
            '-ac', '1',             # Mono
            '-ar', '16000',         # 16kHz sample rate
            '-y', output_path
        ], check=True, capture_output=True)
        print(f"[done]  {output_filename}")
    except subprocess.CalledProcessError as e:
        print(f"[error] {filename}: {e.stderr.decode()}")

print(f"\nFinished. {len(os.listdir(OUTPUT_FOLDER))} WAV file(s) in output folder.")


## ✅ Verify Output
Quick check that the output folder looks right.

In [ ]:
for f in sorted(os.listdir(OUTPUT_FOLDER)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_FOLDER, f)) / (1024 * 1024)
    print(f"  {f} — {size_mb:.1f} MB")
